# 30 — Analysis report (mem_enc)

Checklist A→F. **Không gọi LLM.** Không sửa raw `calls/` / `scores.jsonl`.

| | |
|---|---|
| Artifacts | `results/analysis/` |
| Regenerator | `python scripts/run_analysis.py` |
| Human raw | `data/human/mem_enc_exp1.jsonl` (`human_results`) |
| GPT-4 paper | `data/ready/... gpt4_mean` (≠ `openai/gpt-4.1-mini`) |
| Điều kiện câu | suffix `sample_id`: `all\|global\|animate\|plural\|name` |

Smoke run `gemini-3.6-flash/ORIG` (n=1) bị loại.

In [ ]:
%pip install -q pyyaml pandas matplotlib numpy

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
REPO = None
for c in [HERE, *HERE.parents]:
    if (c / "configs" / "pricing.yaml").exists():
        REPO = c
        break
    if (c / "doan" / "configs" / "pricing.yaml").exists():
        REPO = c / "doan"
        break
assert REPO is not None
sys.path.insert(0, str(REPO / "src"))
print("REPO:", REPO)

In [ ]:
from IPython.display import Image, Markdown, display
import pandas as pd

from plausibility_eval.analysis import run_full_analysis
from plausibility_eval.io_utils import load_yaml
from plausibility_eval.summary import summarize_all

exp = load_yaml(REPO / "configs" / "experiment.yaml")
summary = summarize_all(repo=REPO, coarse_threshold=float(exp.get("coarse_threshold") or 3.0))
print("SUMMARY runs:", summary["n_runs"])

out = run_full_analysis(REPO)
ANALYSIS = Path(out["out_dir"])
print("ANALYSIS:", ANALYSIS)
print("top:", out["findings"].get("top_overall"))

## A — Tổng thể / leaderboard

Chỉ runs `n≥50`. Cột `modes_available` = coverage ablation.

In [ ]:
lb = pd.read_csv(ANALYSIS / "A_leaderboard.csv")
cov = pd.read_json(ANALYSIS / "A_coverage.json", typ="series")
display(lb.sort_values("pearson_r", ascending=False).reset_index(drop=True))
print("Coverage (model → modes):")
print(cov.to_string())
top = lb.iloc[0]
display(Markdown(
    f"**Nhận định A:** Top Pearson = `{top.model_id}` / `{top.mode}` "
    f"(r={top.pearson_r:.4f}, MAE={top.mae:.4f}). "
    f"Model lớn thường chỉ ORIG+T; deepseek & gemma-4 có full ORIG/S/T/ST."
))

## B — Theo điều kiện câu (`all/global/animate/plural/name`)

Heatmap Pearson trên ORIG/T. Rank MAE trung bình qua các run.

In [ ]:
cond = pd.read_csv(ANALYSIS / "B_by_condition.csv")
rank = pd.read_csv(ANALYSIS / "B_condition_rank_mae.csv")
resid = pd.read_csv(ANALYSIS / "B_top_residuals.csv")
display(rank)
display(Image(filename=str(ANALYSIS / "B_condition_heatmap.png")))
display(resid.head(12))
easy, hard = rank.iloc[0], rank.iloc[-1]
display(Markdown(
    f"**Nhận định B:** MAE trung bình thấp nhất ở `{easy.condition}` "
    f"({easy.mean_mae:.3f}); cao nhất `{hard.condition}` ({hard.mean_mae:.3f}). "
    f"Xem residual table để biết câu/model lệch human mạnh nhất."
))

## C — Human disagreement vs LLM dispersion

**Caveat:** human_std = liên-annotator; model_std = liên-sample API (không cùng loại “ý kiến”).

In [ ]:
hi = pd.read_csv(ANALYSIS / "C_high_disagreement_sentences.csv")
disp_sum = pd.read_csv(ANALYSIS / "C_dispersion_summary.csv")
display(hi)
display(disp_sum.sort_values("collapse_rate_on_high_disagreement", ascending=False))
display(Image(filename=str(ANALYSIS / "C_case_histograms.png")))
display(Markdown(
    "**Nhận định C:** Trên top-15 câu human phân tán cao, hầu hết model có "
    "`collapse_rate≈1.0` (model_std ≪ 0.5×human_std): LLM resample ổn định, "
    "không bắt được mức bất đồng của người. Corr(human_std, model_std) thường yếu/âm."
))

## D — Schema làm giảm “giống người”

Chi tiết narrative: `results/analysis/NOTES_D_E.md`.

In [ ]:
dlt = pd.read_csv(ANALYSIS / "D_schema_deltas.csv")
display(dlt)
notes = (ANALYSIS / "NOTES_D_E.md").read_text(encoding="utf-8")
display(Markdown(notes.split("## E —")[0]))
display(Markdown(
    "**Khuyến nghị:** model lớn chỉ chạy **ORIG (+T)**; bỏ S/ST khi mục tiêu là human-likeness."
))

## E — Ranking + trọng điểm GPT-4 paper

So ORIG/T với neo `gpt-4 (paper)` từ ready data.

In [ ]:
rank_e = pd.read_csv(ANALYSIS / "E_ranking_with_gpt4_paper.csv")
overlap = pd.read_csv(ANALYSIS / "E_residual_overlap_vs_gpt4.csv")
g4c = pd.read_json(ANALYSIS / "E_gpt4_paper_by_condition.json")
display(rank_e)
display(Image(filename=str(ANALYSIS / "E_orig_ranking_with_gpt4.png")))
display(Markdown("### GPT-4 paper by condition"))
display(g4c.T)
display(Markdown("### Residual overlap vs GPT-4 (ORIG, |err|≥1)"))
display(overlap)
display(Markdown((ANALYSIS / "NOTES_D_E.md").read_text(encoding="utf-8").split("## E —", 1)[1]))

## F — Chi phí vs người

Giá cập nhật `configs/pricing.yaml` (`as_of: 2026-07-26`, OpenRouter crawl).  
Human baseline ≈ `$0.08 × ~40 annotators` / câu (ước lượng crowdsource, không phải hóa đơn paper).

In [ ]:
cost = pd.read_csv(ANALYSIS / "F_cost_table.csv")
display(cost.sort_values("mean_cost_per_sentence_usd"))
display(Image(filename=str(ANALYSIS / "F_pareto_quality_cost.png")))
best_r = cost.sort_values("pearson_r", ascending=False).iloc[0]
cheapest = cost.sort_values("mean_cost_per_sentence_usd").iloc[0]
display(Markdown(
    f"**Nhận định F:** Mọi run ORIG/T đều `cheaper_than_human=True` dưới baseline crowdsource. "
    f"Rẻ nhất trong bảng: `{cheapest.model_id}`/{cheapest.mode} "
    f"(${cheapest.mean_cost_per_sentence_usd:.4f}/câu). "
    f"Chất lượng cao nhất: `{best_r.model_id}`/{best_r.mode} "
    f"(r={best_r.pearson_r:.3f}, ${best_r.mean_cost_per_sentence_usd:.4f}/câu, "
    f"~{1/best_r.cost_ratio_vs_human:.0f}× rẻ hơn ước lượng human)."
))

## Regenerator

```bash
cd doan && python scripts/run_analysis.py
```

Outputs: `results/analysis/` + refresh `SUMMARY.md` via `summarize_all`.